In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab data access)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Fuzzy C-Means (FCM) Clustering + Kernel Support Vector Machine (Kernel SVM) Pipeline (`models/train_fuzzy_c_means.ipynb`)

This notebook implements a hybrid **Fuzzy C-Means (FCM) Clustering + Kernel Support Vector Machine (Kernel SVM)** classifier for ESI Triage acuity prediction on overlapping clinical vital signs:

### 🔬 Architectural Methodology
1. **3-Way Stratified Data Structure (`Train / Validation / Test`)**:
   - Configured dynamically from `config/triage_conf.json` (`test_size: 0.1` Holdout Test, `val_size: 0.2` Validation, `Train: 70%`).
   - Strict complete-cases filtering on 16 raw features (including `triage_vital_dbp`).
2. **Curated 42-Feature Matrix**:
   - 12 raw vitals/demographics + 30 clinically derived engineered features (thresholds, ranges, ratios, interactions) with `StandardScaler` normalization.
3. **Fuzzy C-Means (FCM) Soft Feature Transformation**:
   - Computes $K$ fuzzy cluster centroids across the high-dimensional clinical manifold.
   - Maps each patient vector $x_i$ into continuous fuzzy membership degrees $U(x_i) = [u_{i1}, u_{i2}, \dots, u_{iK}] \in [0, 1]^K$, capturing fuzzy boundaries and multi-acuity overlaps.
4. **Kernel SVM (Support Vector Machine) Classification**:
   - Feeds the fuzzy membership representations (or hybrid feature space) into a maximum-margin **Kernel SVM** (`SVC(kernel='rbf', class_weight='balanced')`).
   - Maximizes the geometric margin between adjacent and critical ESI classes.
5. **Automated Optuna Hyperparameter Optimization**:
   - Jointly tunes FCM fuzziness $m$, number of clusters $K$, SVM penalty $C$, and kernel coefficient $\gamma$ directly targeting **Validation Set Macro Balanced Accuracy**.
6. **Full Evaluation Suite on Holdout Test Set**:
   - Confusion Matrix, 42 Individual Density Distributions, ROC-AUC Curves, and Optuna Trajectory.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Raw Dataset, Filter Complete Cases & Stratified 3-Way Split (Train/Val/Test)
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(dplyr)
})

config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) config_path <- "config/triage_conf.json"
config <- fromJSON(config_path)

set.seed(config$training$random_state)

stratified_sample <- function(y, fraction, seed = 42) {
  set.seed(seed)
  idx_list <- split(seq_along(y), y)
  sampled <- unlist(lapply(idx_list, function(idx) {
    n_sample <- max(1, round(length(idx) * fraction))
    sample(idx, size = n_sample)
  }))
  return(sort(sampled))
}

data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) data_file <- paste0("../", data_file)

data_env <- new.env()
load(data_file, envir = data_env)

df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)
target_col_name <- config$classes$target_col

initial_total_rows <- nrow(raw_df)
cat("========================================================================\n")
cat(sprintf("  INITIAL DATASET LOADED: %d Total Rows, %d Total Columns\n", initial_total_rows, ncol(raw_df)))
cat("========================================================================\n")
if (target_col_name %in% names(raw_df)) {
  cat("Initial ESI Target Distribution (including NAs):\n")
  print(table(raw_df[[target_col_name]], useNA = "ifany"))
  cat("------------------------------------------------------------------------\n")
}

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(is.na(raw_df$gender), NA, ifelse(as.character(raw_df$gender) == "Male", 1, 0)) else rep(NA, nrow(raw_df))
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) raw_df$cc_breathingdifficulty else rep(NA, nrow(raw_df))

get_vec <- function(col_name) {
  if (col_name %in% names(raw_df)) {
    return(raw_df[[col_name]])
  } else {
    return(rep(NA, nrow(raw_df)))
  }
}

raw_esi <- as.character(raw_df[[target_col_name]])

df_master <- data.frame(
  age                     = raw_df$age,
  cc_breathingdifficulty  = cc_bd_vec,
  gender                  = gender_vec,
  triage_vital_hr         = get_vec("triage_vital_hr"),
  triage_vital_sbp        = get_vec("triage_vital_sbp"),
  triage_vital_dbp        = get_vec("triage_vital_dbp"),
  triage_vital_rr         = get_vec("triage_vital_rr"),
  triage_vital_o2         = get_vec("triage_vital_o2"),
  pulse_min               = get_vec("pulse_min"),
  resp_min                = get_vec("resp_min"),
  spo2_min                = get_vec("spo2_min"),
  sbp_min                 = get_vec("sbp_min"),
  pulse_max               = get_vec("pulse_max"),
  resp_max                = get_vec("resp_max"),
  spo2_max                = get_vec("spo2_max"),
  sbp_max                 = get_vec("sbp_max"),
  target_col              = factor(raw_esi, levels = c("1", "2", "3", "4", "5"))
)

# Strictly drop any row containing at least 1 null/NA value across all 16 raw features or target
df_master <- na.omit(df_master)
df_master$target_num <- as.numeric(as.character(df_master$target_col))

clean_total_rows <- nrow(df_master)
dropped_rows     <- initial_total_rows - clean_total_rows

cat(sprintf("Missing Values Filter: Dropped %d rows with >= 1 NA feature (Retained %d Complete Cases, %.2f%%)\n", 
            dropped_rows, clean_total_rows, (clean_total_rows / initial_total_rows) * 100))
cat("Cleaned ESI Distribution (100% complete cases):\n")
print(table(df_master$target_col))
cat("------------------------------------------------------------------------\n")

# Stratified 3-Way Partitioning dynamically configured from triage_conf.json
test_size <- config$training$test_size
val_size  <- config$training$val_size
seed_val  <- config$training$random_state

# 1. Extract Stratified Holdout Test Set (from config: e.g. 10%)
idx_test <- stratified_sample(df_master$target_col, test_size, seed = seed_val)
test_df_clean  <- df_master[idx_test, ]
rem_df         <- df_master[-idx_test, ]

# 2. Extract Stratified Validation Set from remainder (from config: e.g. 20% of total)
val_adj_fraction <- val_size / (1 - test_size)
idx_val <- stratified_sample(rem_df$target_col, val_adj_fraction, seed = seed_val + 1)
val_df_clean   <- rem_df[idx_val, ]
train_df_clean <- rem_df[-idx_val, ]

train_mat_export <- as.matrix(cbind(train_df_clean[, 1:16], target = train_df_clean$target_num))
val_mat_export   <- as.matrix(cbind(val_df_clean[, 1:16],   target = val_df_clean$target_num))
test_mat_export  <- as.matrix(cbind(test_df_clean[, 1:16],  target = test_df_clean$target_num))

cat(sprintf("3-Way Partition from config/triage_conf.json Complete:\n  Train Set      = %d rows (%.2f%%)\n  Validation Set = %d rows (%.2f%%)\n  Holdout Test   = %d rows (%.2f%%)\n", 
            nrow(train_mat_export), (nrow(train_mat_export) / clean_total_rows) * 100,
            nrow(val_mat_export),   (nrow(val_mat_export) / clean_total_rows) * 100,
            nrow(test_mat_export),  (nrow(test_mat_export) / clean_total_rows) * 100))
cat("========================================================================\n")

In [ ]:
# ---------------------------------------------------------
# Step 2: Build Curated 42-Feature Matrix & Preprocessing
# ---------------------------------------------------------
import os
import pickle
import numpy as np
import pandas as pd
from rpy2.robjects import r
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
import optuna

train_mat_in = np.array(r('train_mat_export'), dtype=np.float64)
val_mat_in   = np.array(r('val_mat_export'),   dtype=np.float64)
test_mat_in  = np.array(r('test_mat_export'),  dtype=np.float64)

raw_mat_tr  = train_mat_in[:, :16]
y_train     = train_mat_in[:, 16].astype(int)

raw_mat_val = val_mat_in[:, :16]
y_val       = val_mat_in[:, 16].astype(int)

raw_mat_ts  = test_mat_in[:, :16]
y_test      = test_mat_in[:, 16].astype(int)

def build_42_feature_matrix(raw_mat):
    N = len(raw_mat)
    X = np.zeros((N, 42), dtype=np.float64)
    
    # Extract raw base columns
    age       = raw_mat[:, 0]
    cc_bd     = raw_mat[:, 1]
    gender    = raw_mat[:, 2]
    t_hr      = raw_mat[:, 3]
    t_sbp     = raw_mat[:, 4]
    t_dbp     = raw_mat[:, 5]
    t_rr      = raw_mat[:, 6]
    t_o2      = raw_mat[:, 7]
    pulse_min = raw_mat[:, 8]
    resp_min  = raw_mat[:, 9]
    spo2_min  = raw_mat[:, 10]
    sbp_min   = raw_mat[:, 11]
    pulse_max = raw_mat[:, 12]
    resp_max  = raw_mat[:, 13]
    spo2_max  = raw_mat[:, 14]
    sbp_max   = raw_mat[:, 15]
    
    hr_rng   = pulse_max - pulse_min
    rr_rng   = resp_max - resp_min
    spo2_rng = spo2_max - spo2_min
    sbp_rng  = sbp_max - sbp_min
    
    # 1..12: Specified Raw Features (including triage_vital_dbp)
    X[:, 0]  = age
    X[:, 1]  = cc_bd
    X[:, 2]  = gender
    X[:, 3]  = t_hr
    X[:, 4]  = t_sbp
    X[:, 5]  = t_dbp
    X[:, 6]  = t_rr
    X[:, 7]  = pulse_min
    X[:, 8]  = resp_min
    X[:, 9]  = spo2_min
    X[:, 10] = pulse_max
    X[:, 11] = spo2_max
    
    # 13..22: Baseline Threshold Flags
    is_dyspnea_tot  = (t_o2 < 90).astype(float)
    is_dyspnea_mod  = ((t_o2 >= 90) & (t_o2 < 94)).astype(float)
    is_brady_pnea   = (t_rr < 10).astype(float)
    is_tachy_pnea   = (t_rr > 30).astype(float)
    is_hypo_tension = (t_sbp <= 90).astype(float)
    is_hyper_tension= (t_sbp > 220).astype(float)
    is_brady_tot    = (t_hr < 40).astype(float)
    is_brady_mod    = ((t_hr >= 40) & (t_hr < 60)).astype(float)
    is_tachy_tot    = (t_hr > 150).astype(float)
    is_tachy_mod    = ((t_hr >= 100) & (t_hr <= 150)).astype(float)
    
    X[:, 12] = is_dyspnea_tot
    X[:, 13] = is_dyspnea_mod
    X[:, 14] = is_brady_pnea
    X[:, 15] = is_tachy_pnea
    X[:, 16] = is_hypo_tension
    X[:, 17] = is_hyper_tension
    X[:, 18] = is_brady_tot
    X[:, 19] = is_brady_mod
    X[:, 20] = is_tachy_tot
    X[:, 21] = is_tachy_mod
    
    # 23..35: Ranges, Mid-to-Triage, Ratios
    X[:, 22] = hr_rng
    X[:, 23] = rr_rng
    X[:, 24] = spo2_rng
    X[:, 25] = sbp_rng
    shock_idx = t_hr / np.where(t_sbp == 0, 1.0, t_sbp)
    X[:, 26] = shock_idx
    X[:, 27] = t_hr - hr_rng
    X[:, 28] = t_sbp - sbp_rng
    X[:, 29] = t_rr - rr_rng
    X[:, 30] = t_o2 - spo2_rng
    X[:, 31] = t_o2 / np.where(t_rr == 0, 1.0, t_rr) # rox_index
    X[:, 32] = spo2_rng / np.where(spo2_max == 0, 1.0, spo2_max) # spo2_drop_ratio
    X[:, 33] = hr_rng / (t_hr + 1.0) # hr_instability_ratio
    X[:, 34] = (t_rr / np.where(t_o2 == 0, 1.0, t_o2)) * 100.0 # bif
    
    # 36..42: Curated Advanced Features
    X[:, 35] = (is_dyspnea_tot + is_dyspnea_mod + is_brady_pnea + is_tachy_pnea +
                is_hypo_tension + is_hyper_tension + is_brady_tot + is_brady_mod +
                is_tachy_tot + is_tachy_mod) # n_abnormal_vitals
    X[:, 36] = (t_hr / 80.0) - (t_sbp / 120.0) # perfusion_gap
    X[:, 37] = np.clip(spo2_min - 90.0, -20.0, 20.0) # resp_reserve
    X[:, 38] = cc_bd * (100.0 - t_o2) # bd_x_o2_deficit
    X[:, 39] = cc_bd * is_tachy_pnea   # bd_x_tachypnea
    X[:, 40] = age * (100.0 - t_o2)   # age_o2_interaction
    X[:, 41] = ((t_hr - 80.0) / 80.0) ** 2 # hr_dev_sq
    
    return X

feature_names_42 = [
    'age', 'cc_breathingdifficulty', 'gender', 'triage_vital_hr', 'triage_vital_sbp', 'triage_vital_dbp', 'triage_vital_rr',
    'pulse_min', 'resp_min', 'spo2_min', 'pulse_max', 'spo2_max',
    'is_dyspnea_total', 'is_dyspnea_moderate', 'is_bradypnea', 'is_tachypnea', 'is_hypotension', 'is_hypertension',
    'is_bradycardia_total', 'is_bradycardia_moderate', 'is_tachycardia_total', 'is_tachycardia_moderate',
    'hr_range', 'rr_range', 'spo2_range', 'sbp_range',
    'shock_index', 'hr_mid_to_triage', 'sbp_mid_to_triage', 'rr_mid_to_triage', 'spo2_mid_to_triage',
    'rox_index', 'spo2_drop_ratio', 'hr_instability_ratio', 'bif',
    'n_abnormal_vitals', 'perfusion_gap', 'resp_reserve', 'bd_x_o2_deficit', 'bd_x_tachypnea',
    'age_o2_interaction', 'hr_dev_sq'
]

X_train_raw = build_42_feature_matrix(raw_mat_tr)
X_val_raw   = build_42_feature_matrix(raw_mat_val)
X_test_raw  = build_42_feature_matrix(raw_mat_ts)

cont_cols_idx = [0, 3, 4, 5, 6, 7, 8, 9, 10, 11, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 36, 37, 38, 40, 41]

scaler = StandardScaler()
X_train = X_train_raw.copy()
X_val   = X_val_raw.copy()
X_test  = X_test_raw.copy()

X_train[:, cont_cols_idx] = scaler.fit_transform(X_train_raw[:, cont_cols_idx])
X_val[:, cont_cols_idx]   = scaler.transform(X_val_raw[:, cont_cols_idx])
X_test[:, cont_cols_idx]  = scaler.transform(X_test_raw[:, cont_cols_idx])

def compute_macro_balanced_accuracy(y_true, y_pred):
    classes = [1, 2, 3, 4, 5]
    bal_accs = []
    for cls in classes:
        y_bin_true = (y_true == cls).astype(int)
        y_bin_pred = (y_pred == cls).astype(int)
        tp = np.sum((y_bin_true == 1) & (y_bin_pred == 1))
        fn = np.sum((y_bin_true == 1) & (y_bin_pred == 0))
        fp = np.sum((y_bin_true == 0) & (y_bin_pred == 1))
        tn = np.sum((y_bin_true == 0) & (y_bin_pred == 0))
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        bal_accs.append((rec + spec) / 2.0)
    return np.mean(bal_accs)

print(f"Curated Feature Matrix Ready: Train={X_train.shape}, Val={X_val.shape}, Test={X_test.shape}")

In [ ]:
# ---------------------------------------------------------
# Step 2.5: Vectorized Fuzzy C-Means Transformer & Kernel SVM Pipeline with Optuna Tuning
# ---------------------------------------------------------
class FuzzyCMeansTransformer:
    """
    Vectorized Fuzzy C-Means (FCM) Clustering Transformer.
    Transforms feature vectors X into soft membership degrees U in [0, 1]^K.
    """
    def __init__(self, n_clusters=5, m=2.0, max_iter=100, tol=1e-4, random_state=42):
        self.n_clusters = n_clusters
        self.m = m
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state
        self.cluster_centers_ = None

    def fit(self, X, y=None):
        np.random.seed(self.random_state)
        N, D = X.shape
        
        # Supervised / Class-Centroid-informed initialization if y is provided
        if y is not None:
            centers = []
            unique_classes = np.unique(y)
            for c in unique_classes:
                mask = (y == c)
                centers.append(np.mean(X[mask], axis=0))
            # If n_clusters > number of unique classes, initialize remaining via stratified sampling
            while len(centers) < self.n_clusters:
                centers.append(X[np.random.choice(N)])
            self.cluster_centers_ = np.array(centers[:self.n_clusters], dtype=np.float64)
        else:
            idx = np.random.choice(N, self.n_clusters, replace=False)
            self.cluster_centers_ = X[idx].copy().astype(np.float64)

        power = 2.0 / (self.m - 1.0)
        for it in range(self.max_iter):
            # Compute Euclidean distances to all centroids: shape (N, K)
            dists = np.zeros((N, self.n_clusters), dtype=np.float64)
            for c in range(self.n_clusters):
                dists[:, c] = np.linalg.norm(X - self.cluster_centers_[c], axis=1)
            dists = np.fmax(dists, 1e-10)
            
            # Update membership matrix U: shape (N, K)
            dists_inv_p = dists ** (-power)
            U = dists_inv_p / np.sum(dists_inv_p, axis=1, keepdims=True)
            
            # Update cluster centroids
            U_m = U ** self.m
            new_centers = np.zeros_like(self.cluster_centers_)
            for c in range(self.n_clusters):
                w_sum = np.sum(U_m[:, c])
                if w_sum > 0:
                    new_centers[c] = np.sum(U_m[:, c:c+1] * X, axis=0) / w_sum
                else:
                    new_centers[c] = self.cluster_centers_[c]
            
            center_shift = np.linalg.norm(new_centers - self.cluster_centers_)
            self.cluster_centers_ = new_centers
            if center_shift < self.tol:
                break
        return self

    def transform(self, X):
        N = len(X)
        power = 2.0 / (self.m - 1.0)
        dists = np.zeros((N, self.n_clusters), dtype=np.float64)
        for c in range(self.n_clusters):
            dists[:, c] = np.linalg.norm(X - self.cluster_centers_[c], axis=1)
        dists = np.fmax(dists, 1e-10)
        dists_inv_p = dists ** (-power)
        U = dists_inv_p / np.sum(dists_inv_p, axis=1, keepdims=True)
        return U

# Create stratified sample for fast, responsive Kernel SVM fitting per trial
np.random.seed(42)
SUB_N = min(15000, len(X_train))
sub_idx_list = []
for c in [1, 2, 3, 4, 5]:
    c_indices = np.where(y_train == c)[0]
    sample_size = int(SUB_N * (len(c_indices) / len(X_train)))
    sub_idx_list.extend(np.random.choice(c_indices, size=max(10, sample_size), replace=False))

sub_idx = np.array(sub_idx_list)
X_train_sub = X_train[sub_idx]
y_train_sub = y_train[sub_idx]

print(f"FCM Clustering Transformer defined. Optuna Trial Sample Size: {len(X_train_sub)} rows")

# Optuna Objective Function: Evaluated strictly on the Validation Set
def fcm_svm_objective(trial):
    n_clusters = trial.suggest_int('n_clusters', 5, 10)
    m = trial.suggest_float('m', 1.2, 2.5)
    feature_mode = trial.suggest_categorical('feature_mode', ['memberships_only', 'hybrid_concat'])
    
    svm_C = trial.suggest_float('svm_C', 0.1, 50.0, log=True)
    svm_gamma = trial.suggest_float('svm_gamma', 0.005, 2.0, log=True)
    
    # 1. Fit FCM on training sample
    fcm = FuzzyCMeansTransformer(n_clusters=n_clusters, m=m, max_iter=60, random_state=42)
    fcm.fit(X_train_sub, y_train_sub)
    
    # 2. Extract FCM Membership Features
    U_train = fcm.transform(X_train_sub)
    U_val   = fcm.transform(X_val)
    
    if feature_mode == 'memberships_only':
        feat_train = U_train
        feat_val   = U_val
    else:
        feat_train = np.hstack([X_train_sub, U_train])
        feat_val   = np.hstack([X_val, U_val])
        
    # 3. Fit Kernel SVM with Balanced Class Weights
    svm = SVC(C=svm_C, kernel='rbf', gamma=svm_gamma, class_weight='balanced', max_iter=2000, random_state=42)
    svm.fit(feat_train, y_train_sub)
    
    # 4. Predict on the Validation Set and evaluate Macro Balanced Accuracy
    preds_val = svm.predict(feat_val)
    val_bal_acc = compute_macro_balanced_accuracy(y_val, preds_val)
    return val_bal_acc

N_TRIALS = 30
print(f"Starting Optuna FCM + Kernel SVM Tuning ({N_TRIALS} Trials, Maximizing Validation Macro Balanced Accuracy)...")
optuna.logging.set_verbosity(optuna.logging.WARNING)
fcm_study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
fcm_study.optimize(fcm_svm_objective, n_trials=N_TRIALS, timeout=None, show_progress_bar=True)

print("========================================================================")
print(f"OPTUNA FCM + KERNEL SVM TUNING COMPLETED!")
print(f"Best Trial #{fcm_study.best_trial.number}: Peak Validation Macro Balanced Accuracy = {fcm_study.best_value:.4f}")
print("Optimal Hyperparameters:", fcm_study.best_trial.params)
print("========================================================================")

In [ ]:
# ---------------------------------------------------------
# Step 2.6: Train Final Production FCM + Kernel SVM Pipeline
# ---------------------------------------------------------
best_params = fcm_study.best_trial.params

print("Training Final Production FCM Transformer & Kernel SVM Model...")
fcm_prod = FuzzyCMeansTransformer(
    n_clusters=best_params['n_clusters'],
    m=best_params['m'],
    max_iter=100,
    random_state=42
)
fcm_prod.fit(X_train, y_train)

U_train_full = fcm_prod.transform(X_train)
U_val_full   = fcm_prod.transform(X_val)
U_test_full  = fcm_prod.transform(X_test)

feature_mode = best_params['feature_mode']
if feature_mode == 'memberships_only':
    feat_train_prod = U_train_full
    feat_val_prod   = U_val_full
    feat_test_prod  = U_test_full
else:
    feat_train_prod = np.hstack([X_train, U_train_full])
    feat_val_prod   = np.hstack([X_val, U_val_full])
    feat_test_prod  = np.hstack([X_test, U_test_full])

# Train Final Kernel SVM on representative stratified balanced training set with probability estimates
svm_prod = SVC(
    C=best_params['svm_C'],
    kernel='rbf',
    gamma=best_params['svm_gamma'],
    class_weight='balanced',
    probability=True,
    random_state=42
)

svm_prod.fit(feat_train_prod[sub_idx], y_train[sub_idx])

# Predictions on Holdout Test Set
preds_test = svm_prod.predict(feat_test_prod)
probs_test = svm_prod.predict_proba(feat_test_prod)

# Export Deployment Bundle
deploy_dir = '../deploy' if os.path.exists('../deploy') else 'deploy'
os.makedirs(deploy_dir, exist_ok=True)

bundle_data = {
    'fcm_transformer': fcm_prod,
    'svm_classifier': svm_prod,
    'best_hyperparameters': best_params,
    'scaler_means': scaler.mean_,
    'scaler_sds': scaler.scale_,
    'cont_cols_idx': cont_cols_idx,
    'feature_names': feature_names_42
}

with open(os.path.join(deploy_dir, 'py_fuzzy_c_means_svm_bundle.pkl'), 'wb') as f:
    pickle.dump(bundle_data, f)

print(f"Fuzzy C-Means + Kernel SVM Bundle saved to: {os.path.join(deploy_dir, 'py_fuzzy_c_means_svm_bundle.pkl')}")

In [ ]:
# ---------------------------------------------------------
# Step 3: Holdout Test Set Evaluation & Detailed Per-Class Breakdown
# ---------------------------------------------------------
def get_per_class_breakdown(y_true, y_pred, probs, pipeline_name):
    classes = [1, 2, 3, 4, 5]
    rows = []
    recalls, specs, bal_accs, aucs = [], [], [], []
    for idx, cls in enumerate(classes):
        y_bin_true = (y_true == cls).astype(int)
        y_bin_pred = (y_pred == cls).astype(int)
        tp = np.sum((y_bin_true == 1) & (y_bin_pred == 1))
        fn = np.sum((y_bin_true == 1) & (y_bin_pred == 0))
        fp = np.sum((y_bin_true == 0) & (y_bin_pred == 1))
        tn = np.sum((y_bin_true == 0) & (y_bin_pred == 0))
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        bal  = (rec + spec) / 2.0
        try: auc = roc_auc_score(y_bin_true, probs[:, idx])
        except Exception: auc = 0.0
        recalls.append(rec); specs.append(spec); bal_accs.append(bal); aucs.append(auc)
        rows.append({
            'Pipeline': pipeline_name,
            'Class': f'ESI_{cls}',
            'Recall': round(rec, 4),
            'Specificity': round(spec, 4),
            'Balanced_Accuracy': round(bal, 4),
            'ROC_AUC': round(auc, 4)
        })
    rows.append({
        'Pipeline': pipeline_name,
        'Class': 'Macro_Average',
        'Recall': round(np.mean(recalls), 4),
        'Specificity': round(np.mean(specs), 4),
        'Balanced_Accuracy': round(np.mean(bal_accs), 4),
        'ROC_AUC': round(np.mean(aucs), 4)
    })
    return pd.DataFrame(rows)

report_df = get_per_class_breakdown(y_test, preds_test, probs_test, 'Fuzzy_C_Means_Kernel_SVM_42feat')

print("========================================================================================")
print("   HOLDOUT TEST SET REPORT: FUZZY C-MEANS + KERNEL SVM PIPELINE (42 FEATURES)")
print("========================================================================================")
print(report_df.to_string(index=False))
print("========================================================================================\n")

reports_dir = '../reports' if os.path.exists('../reports') else 'reports'
os.makedirs(reports_dir, exist_ok=True)

report_df.to_csv(os.path.join(reports_dir, 'fuzzy_c_means_svm_report.csv'), index=False)
print(f"Report saved to {os.path.join(reports_dir, 'fuzzy_c_means_svm_report.csv')}")

In [ ]:
# ---------------------------------------------------------
# Step 4: Confusion Matrix Graph for Holdout Test Benchmark
# ---------------------------------------------------------
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

plots_dir = '../plots' if os.path.exists('../plots') else 'plots'
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'image'), exist_ok=True)

esi_labels = [f"ESI {i}" for i in range(1, 6)]

# Compute confusion matrix
cm_svm      = confusion_matrix(y_test, preds_test, labels=[1, 2, 3, 4, 5])
cm_svm_norm = cm_svm.astype('float') / cm_svm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(9, 7))

annot_svm = np.empty_like(cm_svm, dtype=object)
for i in range(5):
    for j in range(5):
        annot_svm[i, j] = f"{cm_svm[i, j]}\n({cm_svm_norm[i, j]*100:.1f}%)"

sns.heatmap(cm_svm_norm, annot=annot_svm, fmt='', cmap='Purples', cbar=True,
            xticklabels=esi_labels, yticklabels=esi_labels, ax=ax, vmin=0, vmax=1)
ax.set_title('Holdout Test Confusion Matrix\nFuzzy C-Means + Kernel SVM (42 Features)', fontsize=12.5, fontweight='bold', pad=12)
ax.set_xlabel('Predicted ESI Level', fontsize=11, fontweight='bold')
ax.set_ylabel('True ESI Level', fontsize=11, fontweight='bold')

plt.tight_layout()

cm_path = os.path.join(plots_dir, 'holdout_test_confusion_matrix_fcm_svm.png')
plt.savefig(cm_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'holdout_test_confusion_matrix_fcm_svm.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"Confusion Matrix Graph saved to: {cm_path}")

In [ ]:
# ---------------------------------------------------------
# Step 5: Density Data Distribution Graphs (Saved to Individual PNG per Feature)
# ---------------------------------------------------------
import os
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

plots_dir = '../plots' if os.path.exists('../plots') else 'plots'
density_dir = os.path.join(plots_dir, 'density_fcm_svm')
density_img_dir = os.path.join(plots_dir, 'image', 'density_fcm_svm')
os.makedirs(density_dir, exist_ok=True)
os.makedirs(density_img_dir, exist_ok=True)

# Construct full DataFrame of 42 features and true ESI labels
df_raw_all = pd.DataFrame(X_train_raw, columns=feature_names_42)
df_raw_all['ESI'] = [f"ESI {k}" for k in y_train]

esi_palette = {
    'ESI 1': '#d62728',  # Red (Resuscitation)
    'ESI 2': '#ff7f0e',  # Orange (Emergent)
    'ESI 3': '#2ca02c',  # Green (Urgent)
    'ESI 4': '#1f77b4',  # Blue (Less Urgent)
    'ESI 5': '#9467bd'   # Purple (Non-urgent)
}

print(f"Saving individual feature density distribution plots ({len(feature_names_42)} features) to: {density_dir}")

for feat in feature_names_42:
    fig, ax = plt.subplots(figsize=(8, 5))
    
    if feat in ['gender', 'cc_breathingdifficulty'] or feat.startswith('is_') or feat.startswith('bd_x_tachypnea'):
        prop_df = df_raw_all.groupby('ESI')[feat].mean().reset_index(name='Proportion')
        sns.barplot(data=prop_df, x='ESI', y='Proportion', palette=esi_palette, ax=ax, edgecolor='black')
        ax.set_title(f"{feat} (Prevalence by ESI Level)", fontsize=13, fontweight='bold', pad=12)
        ax.set_xlabel("ESI Level", fontsize=11, fontweight='bold')
        ax.set_ylabel("Prevalence / Proportion", fontsize=11, fontweight='bold')
        for p in ax.patches:
            ax.annotate(f"{p.get_height()*100:.1f}%",
                        (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='bottom', fontsize=10, fontweight='bold', xytext=(0, 2),
                        textcoords='offset points')
    else:
        sns.kdeplot(
            data=df_raw_all,
            x=feat,
            hue='ESI',
            palette=esi_palette,
            common_norm=False,
            fill=True,
            alpha=0.20,
            linewidth=2.0,
            ax=ax
        )
        ax.set_title(f"Feature Density Distribution: {feat}", fontsize=13, fontweight='bold', pad=12)
        ax.set_xlabel(feat, fontsize=11, fontweight='bold')
        ax.set_ylabel("Density", fontsize=11, fontweight='bold')
    
    ax.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    
    out_file = f"density_{feat}.png"
    plt.savefig(os.path.join(density_dir, out_file), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(density_img_dir, out_file), dpi=300, bbox_inches='tight')
    plt.close()

print(f"All {len(feature_names_42)} individual density plots successfully saved in {density_dir}")

In [ ]:
# ---------------------------------------------------------
# Step 6: Multiclass ROC-AUC Curve Analysis (Holdout Test Benchmark)
# ---------------------------------------------------------
import os
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

plots_dir = '../plots' if os.path.exists('../plots') else 'plots'
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'image'), exist_ok=True)

classes = [1, 2, 3, 4, 5]
y_test_bin = label_binarize(y_test, classes=classes)
n_classes  = len(classes)

fpr = dict()
tpr = dict()
roc_auc = dict()

for i, cls in enumerate(classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], probs_test[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

fpr["micro"], tpr["micro"], _ = roc_curve(y_test_bin.ravel(), probs_test.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(n_classes):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
mean_tpr /= n_classes

fpr["macro"] = all_fpr
tpr["macro"] = mean_tpr
roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

plt.figure(figsize=(9, 8))

esi_colors = {
    0: '#d62728',  # ESI 1: Red
    1: '#ff7f0e',  # ESI 2: Orange
    2: '#2ca02c',  # ESI 3: Green
    3: '#1f77b4',  # ESI 4: Blue
    4: '#9467bd'   # ESI 5: Purple
}

plt.plot(fpr["micro"], tpr["micro"],
         label=f"Micro-Average (AUC = {roc_auc['micro']:.4f})",
         color='#e377c2', linestyle=':', linewidth=2.5)
plt.plot(fpr["macro"], tpr["macro"],
         label=f"Macro-Average (AUC = {roc_auc['macro']:.4f})",
         color='#17becf', linestyle='--', linewidth=2.5)

for i, cls in enumerate(classes):
    plt.plot(fpr[i], tpr[i], color=esi_colors[i], linewidth=2.0,
             label=f"ESI {cls} (AUC = {roc_auc[i]:.4f})")

plt.plot([0, 1], [0, 1], 'k--', color='gray', linewidth=1.2, label='Random Guess (AUC = 0.5000)')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12, fontweight='bold')
plt.ylabel('True Positive Rate (Recall / Sensitivity)', fontsize=12, fontweight='bold')
plt.title('Holdout Test ROC-AUC Curves\nFuzzy C-Means + Kernel SVM (42 Features)', fontsize=14, fontweight='bold', pad=12)
plt.legend(loc="lower right", fontsize=10.5, frameon=True, framealpha=0.95)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

roc_plot_path = os.path.join(plots_dir, 'holdout_test_roc_auc_curve_fcm_svm.png')
plt.savefig(roc_plot_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'holdout_test_roc_auc_curve_fcm_svm.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"ROC-AUC Curve Graph saved to: {roc_plot_path}")

In [ ]:
# ---------------------------------------------------------
# Step 7: Optuna FCM + Kernel SVM Optimization Trajectory Graph
# ---------------------------------------------------------
import matplotlib.pyplot as plt
import seaborn as sns

trial_nums = [t.number for t in fcm_study.trials if t.value is not None]
trial_vals = [t.value for t in fcm_study.trials if t.value is not None]
running_max = np.maximum.accumulate(trial_vals)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# 1. Optimization Trajectory
ax1.scatter(trial_nums, trial_vals, color='#9467bd', alpha=0.7, label='Trial Macro Balanced Acc')
ax1.plot(trial_nums, running_max, color='#d62728', linewidth=2.2, label='Peak Macro Balanced Acc')
ax1.set_title('Optuna FCM + Kernel SVM Search History', fontsize=12.5, fontweight='bold', pad=12)
ax1.set_xlabel('Trial Number', fontsize=11, fontweight='bold')
ax1.set_ylabel('Val Macro Balanced Accuracy', fontsize=11, fontweight='bold')
ax1.legend(loc='lower right')
ax1.grid(True, linestyle='--', alpha=0.4)

# 2. FCM Centroid Mean Absolute Norm per Cluster
centroid_magnitudes = np.linalg.norm(fcm_prod.cluster_centers_, axis=1)
cluster_names = [f'Cluster {k+1}' for k in range(len(centroid_magnitudes))]

sns.barplot(x=cluster_names, y=centroid_magnitudes, palette='viridis', ax=ax2, edgecolor='black')
ax2.set_title(f'FCM Learned Cluster Centroid Magnitudes (K={len(centroid_magnitudes)}, m={best_params["m"]:.2f})', fontsize=12.5, fontweight='bold', pad=12)
ax2.set_xlabel('Fuzzy Cluster ID', fontsize=11, fontweight='bold')
ax2.set_ylabel('Centroid L2 Norm', fontsize=11, fontweight='bold')
for p in ax2.patches:
    ax2.annotate(f"{p.get_height():.2f}",
                 (p.get_x() + p.get_width() / 2., p.get_height()),
                 ha='center', va='bottom', fontsize=10.5, fontweight='bold', xytext=(0, 2),
                 textcoords='offset points')
ax2.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
traj_plot_path = os.path.join(plots_dir, 'optuna_fcm_svm_trajectory.png')
plt.savefig(traj_plot_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'optuna_fcm_svm_trajectory.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"Optimization Trajectory Graph saved to: {traj_plot_path}")